In [20]:
from pathlib import Path
import sys
import polars as pl
import numpy as np
import pandas as pd
import bottleneck as bn
import bisect
from datetime import datetime, timedelta
from dataclasses import dataclass
import pymssql
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [21]:
zyyx_url = URL.create(drivername="mssql+pymssql",
             username="zyyxReader",
             password="zyyx!5893@Fund",
             host="10.110.0.106",
             database="zyyx",
             query={"charset": "utf8"})
# 连接时显式指定 tds_version 和 UTF-8 字符集
zyyx_engine = create_engine(
    zyyx_url,
    connect_args={
        "tds_version": "7.0",
        "charset": "utf8"
    }
)

conn = zyyx_engine.connect()

In [22]:
JY_CONFIG = {
    "server": "10.10.0.102",
    "user": "jydbReader",
    "password": "jy@9043!Reader",
    "database": "jydb",
    "charset": "cp936",
}
jy_conn = pymssql.connect(**JY_CONFIG)

In [8]:
@dataclass(frozen=True)
class AnalystFactorConfig:
    lookback_days: int = 90
    max_entry_delay_days: int = 7
    min_reliability: int = 5
    min_dispersion_orgs: int = 5
    revision_gap_days: int = 30
    revision_history_days: int = 180
    min_profit_denominator: float = 100.0


def _date(x):
    x = pd.Timestamp(x).normalize()
    if pd.isna(x): raise ValueError("invalid date")
    return x


def load_stock_marketinfo(date,ticks,jy_conn):
    '''获取股票市值/股本/收盘价信息'''
    sql=f"""
    SELECT
        C.SecuCode as stock_code,
        A.ClosePrice as close,
        A.TotalMV as total_mv,
        A.TotalMV/NULLIF(A.ClosePrice,0) as total_share
    FROM QT_StockPerformance A
    JOIN SecuMain C ON C.InnerCode=A.InnerCode
    WHERE A.TradingDay='{date}' AND C.SecuMarket IN (83,90) AND C.SecuCategory=1
    UNION ALL
    SELECT
        C.SecuCode as stock_code,
        B.ClosePrice as close,
        B.TotalMV as total_mv,
        B.TotalMV/NULLIF(B.ClosePrice,0) as total_share
    FROM LC_STIBPerformance B
    JOIN SecuMain C ON C.InnerCode=B.InnerCode
    WHERE B.TradingDay='{date}' AND C.SecuMarket IN (83,90) AND C.SecuCategory=1
    """
    out=pd.read_sql(sql,jy_conn)
    out["stock_code"]=out.stock_code.astype("string").str.zfill(6)
    return out.set_index('stock_code').reindex([t for t in ticks if t!=''])



def load_annual_actuals(fy,date,ticks,jy_conn):
    """Load annual actual profit and align it to the stock axis."""
    asof=_date(date)
    sql=f"""
    WITH ranked AS (
        SELECT
            S.SecuCode AS stock_code,
            A.EndDate AS end_date,
            A.InfoPublDate AS info_pub_date,
            A.NetProfit/10000.0 AS actual_np,
            ROW_NUMBER() OVER (
                PARTITION BY S.SecuCode,A.EndDate
                ORDER BY
                    CASE WHEN A.BulletinType=20 THEN 0 ELSE 1 END,
                    A.InfoPublDate,
                    A.ID
            ) AS rn
        FROM LC_IncomeStatementAll A
        JOIN SecuMain S ON S.CompanyCode=A.CompanyCode
        WHERE A.EndDate='{int(fy)}-12-31'
          AND A.InfoPublDate<='{asof.date()}'
          AND A.IfMerged=1
          AND A.IfAdjusted=2
          AND A.IfComplete=1
          AND A.BulletinType IN (20,30)
          AND S.SecuCategory=1
          AND S.SecuMarket IN (83,90)
    )
    SELECT stock_code,end_date,info_pub_date,actual_np
    FROM ranked
    WHERE rn=1
    """
    actual=pl.read_database(
        sql,jy_conn,infer_schema_length=None
    ).with_columns(
        pl.col("stock_code").cast(pl.String).str.zfill(6),
        pl.col("actual_np").cast(pl.Float64,strict=False),
        pl.lit(int(fy)).alias("report_year"),
    )
    stocks=[str(t).zfill(6) for t in ticks if t!=""]
    return pl.DataFrame({
        "stock_code":stocks,
        "_order":range(len(stocks)),
    }).join(
        actual,on="stock_code",how="left"
    ).sort("_order").drop("_order")

def load_forecasts(fy,conn,date,cfg,latest_only=False):
    """Load annual forecasts, optionally keeping the latest row per stock and institution."""
    asof=_date(date)
    sql=f"""
    SELECT
        f.id,
        f.report_id,
        f.stock_code,
        f.organ_id,
        f.create_date,
        f.entrytime,
        f.report_year,
        f.report_quarter,
        f.forecast_np
    FROM rpt_forecast_stk f
    WHERE f.report_year={int(fy)}
      AND f.report_quarter=4
      AND f.entrytime<='{asof.date()}'
      AND f.create_date<='{asof.date()}'
      AND DATEDIFF(day,f.create_date,f.entrytime) BETWEEN 0 AND {cfg.max_entry_delay_days}
      AND f.reliability>={cfg.min_reliability}
    """
    out=(
        pl.read_database(sql,conn,infer_schema_length=None)
        .with_columns(
            pl.col("stock_code").cast(pl.String).str.zfill(6),
            pl.col("create_date").cast(pl.Datetime,strict=False),
            pl.col("entrytime").cast(pl.Datetime,strict=False),
            pl.col("forecast_np").cast(pl.Float64,strict=False),
        )
        .filter(
            pl.col("stock_code").str.contains(r"^(00|30|60|68|92)")
            &pl.col("forecast_np").is_not_null()
            &pl.col("entrytime").is_not_null()
        )
        .sort(["stock_code","organ_id","entrytime","report_id","id"])
    )
    if latest_only:
        out=out.unique(
            subset=["stock_code","organ_id"],
            keep="last",
            maintain_order=True,
        )
    return out

def calc_afe(date,ticks,conn,jy_conn,cfg):
    asof=_date(date)
    fy=asof.year-1 if (asof.month,asof.day)>=(5,1) else asof.year-2
    actual=load_annual_actuals(fy,date,ticks,jy_conn)
    forecast=load_forecasts(fy,conn,date,cfg)
    return forecast.join(
        actual.select("stock_code","actual_np","info_pub_date"),
        on="stock_code",
        how="left",
    ).filter(
        pl.col("create_date")<pl.col("info_pub_date")
    ).sort(
        ["stock_code","organ_id","entrytime","report_id","id"]
    ).unique(
        subset=["stock_code","organ_id"],keep="last",maintain_order=True
    ).with_columns(
        (pl.col("forecast_np")-pl.col("actual_np")).abs().alias("afe")
    )

def calc_pafe(date,ticks,conn,jy_conn,cfg):
    pafe=calc_afe(date,ticks,conn,jy_conn,cfg)
    mean_afe=pl.col("afe").mean().over("stock_code")
    return pafe.with_columns(
        ((pl.col("afe")-mean_afe)/mean_afe).alias("pafe")
    )

def calc_last_accwt(date,ticks,conn,jy_conn,cfg):
    accwt=calc_pafe(date,ticks,conn,jy_conn,cfg)
    mean_pafe=pl.col("pafe").mean().over("stock_code")
    std_pafe=pl.col("pafe").std().over("stock_code")
    accwt=accwt.with_columns(
        ((pl.col("pafe")-mean_pafe)/std_pafe).alias("pafe_z")
    ).with_columns(
        pl.when(pl.col("pafe_z")<0)
        .then(-pl.col("pafe_z"))
        .otherwise(0.0)
        .alias("accwt")
    )
    return accwt.with_columns(
        (pl.col("accwt")/pl.col("accwt").sum().over("stock_code")).alias("accwt")
    )


def calc_con_forecast(date,ticks,conn,jy_conn,cfg):
    weight=calc_last_accwt(
        date,ticks,conn,jy_conn,cfg
    ).select("stock_code","organ_id","accwt")
    asof=_date(date)
    fy1=asof.year if (asof.month,asof.day)>=(5,1) else asof.year-1
    forecast=load_forecasts(
        fy1,conn,date,cfg,latest_only=True
    ).filter(
        pl.col("entrytime")>=pl.lit(
            asof.to_pydatetime()-pd.Timedelta(days=cfg.lookback_days))
    )
    return forecast.join(
        weight,on=["stock_code","organ_id"],how="inner"
    ).with_columns(
        (pl.col("accwt")/pl.col("accwt").sum().over("stock_code")).alias("accwt")
    ).with_columns(
        (pl.col("forecast_np")*pl.col("accwt")).alias("weighted_forecast")
    ).group_by("stock_code").agg(
        pl.col("weighted_forecast").sum().alias("con_forecast"),
        pl.len().alias("num_forecasts"),
    )

In [23]:
ticks = np.load("D:\data\\axis\\ticks.npy", allow_pickle=True)
date = '2024-06-30'

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\xujiayi\AppData\Local\Temp\ipykernel_22084\2329426862.py:1: SyntaxWarning: invalid escape sequence '\d'
  ticks = np.load("D:\data\\axis\\ticks.npy", allow_pickle=True)
